<a href="https://colab.research.google.com/github/fboldt/aulas-am-bsi/blob/main/aula20a_sele%C3%A7%C3%A3o_de_caracter%C3%ADsticas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

dataset = fetch_openml('arrhythmia', version=1)
X, y = dataset.data, dataset.target
imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(X_train.shape, X_test.shape)

(361, 279) (91, 279)


In [55]:
from sklearn.neighbors import KNeighborsClassifier

model = KNeighborsClassifier()

def evaluate_model(model, data):
    X_train, X_test, y_train, y_test = data
    model.fit(X_train, y_train)
    return model.score(X_test, y_test)

data = (X_train, X_test, y_train, y_test)
print(evaluate_model(model, data))

0.5714285714285714


In [56]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

normalization = StandardScaler()
model_std = make_pipeline(normalization, model)
print(evaluate_model(model_std, data))

0.5494505494505495


#PCA

In [57]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.99)
model_pca = make_pipeline(pca, model)
print(evaluate_model(model_pca, data))

0.5714285714285714


In [58]:
n_feats = model_pca.steps[0][1].n_components_
print(n_feats)

63


In [59]:
model_std_pca = make_pipeline(normalization, pca, model)
print(evaluate_model(model_std_pca, data))

0.5384615384615384


In [60]:
model_pca_std = make_pipeline(pca, normalization, model)
print(evaluate_model(model_pca_std, data))

0.4835164835164835


# Feature Selection
## Ranking
### Filter

In [61]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin

def variance_score(X, y=None):
    return np.var(X, axis=0)[0]

def feature_ranking(X, y=None, score_function=variance_score):
    scores = []
    for f in range(X.shape[1]):
        score = score_function(X[:, f:f+1], y)
        scores.append(score)
    return np.argsort(scores)[::-1]

class FeatureRanking(BaseEstimator, TransformerMixin):
    def __init__(self, n_feats=None,
                 score_function=variance_score,
                 ranking_function=feature_ranking):
        self.n_feats = n_feats
        self.score_function = score_function
        self.ranking_function = ranking_function
    def fit(self, X, y=None):
        if self.n_feats is None:
            self.feats = X.shape[1]
        ranking = self.ranking_function(X, y, self.score_function)
        self.feats = ranking[:self.n_feats]
        return self
    def transform(self, X, y=None):
        return X[:, self.feats]

feat_rank = FeatureRanking(n_feats=n_feats)
feat_rank.fit(X_train, y_train)
print(feat_rank.feats)
model_fr = make_pipeline(feat_rank, model)
print(evaluate_model(model_fr, data))

[ 10  13   9   5   2   7 248  12 258   6 247  11  56  87 238  51   8  41
 237 268 257  89  40  64  65  77 149  39  99  29 227  76  17  53 278  16
  44 178  92  63  28 188 101 228 137  52   3 125   0 267 218 187 113   4
  75  68 111 148 177 100 217  88 277]
0.5494505494505495


In [62]:
class HybridSelection(BaseEstimator, TransformerMixin):
    def __init__(self, model,
                 score_function=variance_score,
                 ranking_function=feature_ranking):
        self.model = model
        self.score_function = score_function
        self.ranking_function = ranking_function
    def fit(self, X, y=None):
        X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
        ranking = self.ranking_function(X_tr, y_tr, self.score_function)
        best_score = -np.inf
        best_n = 0
        for n in range(1, X_tr.shape[1]+1):
            feats = ranking[:n]
            model = self.model.fit(X_tr[:, feats], y_tr)
            score = model.score(X_val[:, feats], y_val)
            if score > best_score:
                best_score = score
                best_n = n
        self.feats = ranking[:best_n]
        return self
    def transform(self, X, y=None):
        return X[:, self.feats]

hybr_rank = HybridSelection(model)
hybr_rank.fit(X_train, y_train)
print(len(hybr_rank.feats))
print(hybr_rank.feats)

21
[ 10  13   5   2   9 248   7  12 258   6 247  11  56  87 237 257   8 268
 238  41  51]


In [63]:
model_hr = make_pipeline(hybr_rank, model)
model_hr.fit(X_train, y_train)
print(f"Resubstituition: {model_hr.score(X_train, y_train)}")#
print(f"Holdout: {evaluate_model(model_hr, data)}")

Resubstituition: 0.6620498614958449
Holdout: 0.5494505494505495


In [64]:
import numpy as np

TMP = np.array([
    [1, 2, 3],
    [0, 1, 1],
    [4, 5, 6],
    [1, 4, 1],
    [7, 8, 9],
    [6, 1, 8],
])

def pairwise_distances(X1, X2=None):
  if X2 is None:
    X2 = X1
  return np.linalg.norm(X1[:, None, :] - X2[None, :, :], axis=-1)

print(pairwise_distances(TMP))

[[ 0.          2.44948974  5.19615242  2.82842712 10.39230485  7.14142843]
 [ 2.44948974  0.          7.54983444  3.16227766 12.72792206  9.21954446]
 [ 5.19615242  7.54983444  0.          5.91607978  5.19615242  4.89897949]
 [ 2.82842712  3.16227766  5.91607978  0.         10.77032961  9.11043358]
 [10.39230485 12.72792206  5.19615242 10.77032961  0.          7.14142843]
 [ 7.14142843  9.21954446  4.89897949  9.11043358  7.14142843  0.        ]]


In [65]:
for f in range(TMP.shape[1]):
  print(f"Feature: {f}")
  dists = pairwise_distances(TMP[:, f:f+1])
  print(dists)
  print(f"mean distance: {dists.mean()}")

Feature: 0
[[0. 1. 3. 0. 6. 5.]
 [1. 0. 4. 1. 7. 6.]
 [3. 4. 0. 3. 3. 2.]
 [0. 1. 3. 0. 6. 5.]
 [6. 7. 3. 6. 0. 1.]
 [5. 6. 2. 5. 1. 0.]]
mean distance: 2.9444444444444446
Feature: 1
[[0. 1. 3. 2. 6. 1.]
 [1. 0. 4. 3. 7. 0.]
 [3. 4. 0. 1. 3. 4.]
 [2. 3. 1. 0. 4. 3.]
 [6. 7. 3. 4. 0. 7.]
 [1. 0. 4. 3. 7. 0.]]
mean distance: 2.7222222222222223
Feature: 2
[[0. 2. 3. 2. 6. 5.]
 [2. 0. 5. 0. 8. 7.]
 [3. 5. 0. 5. 3. 2.]
 [2. 0. 5. 0. 8. 7.]
 [6. 8. 3. 8. 0. 1.]
 [5. 7. 2. 7. 1. 0.]]
mean distance: 3.5555555555555554


In [66]:
def intra_class_mean_distance(X, y):
  mean_dist = 0
  labels = np.unique(y)
  for label in labels:
    X_label = X[y==label]
    dists = pairwise_distances(X_label)
    mean_dist += dists.mean()
  return mean_dist / len(labels)

y_tmp = np.array([0,1,0,1,0,1])
print(intra_class_mean_distance(TMP, y_tmp))

4.697429487492455


In [67]:
def inter_class_mean_distance(X, y):
  mean_dist = 0
  labels = np.unique(y)
  for label in labels:
    in_class = X[y==label]
    out_class = X[y!=label]
    dists = pairwise_distances(in_class, out_class)
    mean_dist += dists.mean()
  return mean_dist / len(labels)

print(inter_class_mean_distance(TMP, y_tmp))

6.824879900464295


In [68]:
def distance_score(X, y):
  return inter_class_mean_distance(X, y) / (intra_class_mean_distance(X, y)+1e-8)

print(distance_score(TMP, y_tmp))

1.452896718110937


In [69]:
feat_rank_dist = HybridSelection(model, score_function=distance_score)
feat_rank_dist.fit(X_train, y_train)
print(len(feat_rank_dist.feats))
print(feat_rank_dist.feats)

55
[ 48 150  23  37  47  59 153  45 155  71  73  85  97  84 130 273  43 113
   5  16 184   4 243 111 148 237   8  41 229 227  39  18 114  52  56 277
  27  14 275 265  90 170  20 247 236  51   7  66 112  95 163  28 257  63
   6]


In [70]:
model_hrd = make_pipeline(feat_rank_dist, model)
print(evaluate_model(model_hrd, data))

0.5934065934065934


### Wrapper

In [71]:
def wrapper_score(X, y):
  X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
  model.fit(X_tr, y_tr)
  return model.score(X_val, y_val)

feat_rank_rapper = HybridSelection(model, score_function=wrapper_score)
feat_rank_rapper.fit(X_train, y_train)
print(len(feat_rank_rapper.feats))
print(feat_rank_rapper.feats)

140
[227 240 223 242  90 113 112 256 124 241  75 206 210 246 251 230 191 250
  27 248  20 101 123 148 152  14 266 128  16 170  80 182  22   0 278   3
  21 252  35 244 245  26  29  32  34  23  24 216 214  49  45 239 229  69
  68  67  60  59  58  57  73  83  81 198 199 204  77  72  71 208 205 200
 184 220 224  52  53  55 238 234 243  41  43 257 254  96  95  93  92  88
  86  85  84 180 194 192 185 179 178 168 164 108 109 115 114 150 151 153
 154 155 156 174 105 107 106 167 165 120 119 118 160 158 157 127 121 132
 134 135 138 139 129 130 131 143 142 141 144   5 146  10]


In [72]:
model_frw = make_pipeline(feat_rank_rapper, model)
print(evaluate_model(model_frw, data))

0.6153846153846154


# SFS - Sequential Forward Selection

In [73]:
def sequential_forward_selection(X, y=None, score_function=variance_score):
    scores = []
    best_combination = []
    remaining_features = list(range(X.shape[1]))
    for _ in range(X.shape[1]):
        best_score = -np.inf
        best_feature = None
        for f in remaining_features:
            feats = best_combination + [f]
            score = score_function(X[:, feats], y)
            if score > best_score:
                best_score = score
                best_feature = f
        scores.append(best_score)
        best_combination.append(best_feature)
        remaining_features.remove(best_feature)
    return np.argsort(scores)[::-1]

feat_rank = FeatureRanking(n_feats=n_feats,
                           ranking_function=sequential_forward_selection)
feat_rank.fit(X_train, y_train)
print(feat_rank.feats)

[  0 278 277 276 275 274 273  16  17  18  19  20  21  22  23  24  25  26
  27  28  29  30  31 240 241 242 243 244 245 246 247 248 249 250 251 252
 253 254 255 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238
 239 208 209 210 211 212 213 214 215]


In [74]:
hybr_rank = HybridSelection(model, ranking_function=sequential_forward_selection)
hybr_rank.fit(X_train, y_train)
print(len(hybr_rank.feats))
print(hybr_rank.feats)

254
[  0 278 277 276 275 274 273  16  17  18  19  20  21  22  23  24  25  26
  27  28  29  30  31 240 241 242 243 244 245 246 247 248 249 250 251 252
 253 254 255 224 225 226 227 228 229 230 231 232 233 234 235 236 237 238
 239 208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223 192
 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207 176 177 178
 179 180 181 182 183 184 185 186 187 188 189 190 191 160 161 162 163 164
 165 166 167 168 169 170 171 172 173 174 175 144 145 146 147 148 149 150
 151 152 153 154 155 156 157 158 159 128 129 130 131 132 133 134 135 136
 137 138 139 140 141 142 143 112 113 114 115 116 117 118 119 120 121 122
 123 124 125 126 127  96  97  98  99 100 101 102 103 104 105 106 107 108
 109 110 111  80  81  82  83  84  85  86  87  88  89  90  91  92  93  94
  95  64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79  48
  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63  32  33  34
  35  36  37  38  39  40  41  42  43  44  45  4

In [75]:
model_hr = make_pipeline(hybr_rank, model)
model_hr.fit(X_train, y_train)
print(f"Resubstituition: {model_hr.score(X_train, y_train)}")#
print(f"Holdout: {evaluate_model(model_hr, data)}")

Resubstituition: 0.6814404432132964
Holdout: 0.6263736263736264


In [76]:
print(len(model_hr.steps[0][1].feats))

254


In [77]:
feat_sfs_dist = HybridSelection(model, score_function=distance_score,
                                 ranking_function=sequential_forward_selection)
# model_hrd = make_pipeline(feat_sfs_dist, model)
# print(evaluate_model(model_hrd, data))
# print(len(model_hrd.steps[0][1].feats))

In [78]:
feat_sfs_rapper = HybridSelection(model, score_function=wrapper_score,
                                   ranking_function=sequential_forward_selection)
model_frw = make_pipeline(feat_sfs_rapper, model)
print(evaluate_model(model_frw, data))
print(len(model_frw.steps[0][1].feats))

0.5054945054945055
31


In [79]:
feat_rank_dist = HybridSelection(model, score_function=distance_score)
feat_sfs_dist = HybridSelection(model, ranking_function=sequential_forward_selection)
model_rank_sfs = make_pipeline(feat_rank_dist, feat_sfs_dist, model)
print(evaluate_model(model_rank_sfs, data))
print(len(model_rank_sfs.steps[0][1].feats), len(model_rank_sfs.steps[1][1].feats))

0.5934065934065934
55 38


In [82]:
from sklearn.model_selection import cross_validate, KFold
from pprint import pprint

scores = cross_validate(model_rank_sfs, X, y, cv=KFold(n_splits=5), return_train_score=True)
pprint(scores)

{'fit_time': array([3.09845376, 2.45460582, 2.29497623, 2.67991853, 3.31134343]),
 'score_time': array([0.0022037 , 0.00229716, 0.00296307, 0.00295544, 0.00303316]),
 'test_score': array([0.53846154, 0.67032967, 0.62222222, 0.5       , 0.7       ]),
 'train_score': array([0.67867036, 0.64542936, 0.6878453 , 0.66850829, 0.6878453 ])}


In [83]:
print(f"mean train: {scores['train_score'].mean()}")
print(f"std train: {scores['train_score'].std()}")
print(f"mean test: {scores['test_score'].mean()}")
print(f"std test: {scores['test_score'].std()}")
print(f"mean time: {scores['fit_time'].mean()}")
print(f"std time: {scores['fit_time'].std()}")

mean train: 0.6736597236038628
std train: 0.0158180808927474
mean test: 0.6062026862026861
std test: 0.07620328779921463
mean time: 2.7678595542907716
std time: 0.38318335441481727
